# 문항 1 : 전자상거래 회원·상품 데이터 모델링 및 타입 안전 집계 유틸리티 구현

## [필수]
- 인터페이스와 타입 별칭으로 회원(User)과 상품(Product) 데이터 구조를 정의
- 더미 데이터로 타입 규칙을 검증

In [1]:
// 회원 모델 정의

type UserRole = "admin" | "customer" | "guest";

interface User {
  id: number;
  name: string;
  email: string;
  role: UserRole;
  isActive: boolean;
}

In [2]:
// 상품 모델 정의

type Category = "electronics" | "clothing" | "books";

interface Product {
  id: number;
  title: string;
  price: number;
  category: Category;
  stock: number;
  discountRate?: number; // 선택적 프로퍼티
}

In [3]:
// 데이터 바인딩 및 검증

const users: User[] = [
    { 
        id: 1,
        name: "관리자",
        email: "admin@shop.com",
        role: "admin",
        isActive: true
    },
    {
        id: 2,
        name: "고객",
        email: "customer@shop.com",
        role: "customer",
        isActive: true
    }, 
    {
        id: 3,
        name: "게스트",
        email: "guest@shop.com",
        role: "guest",
        isActive: true
    }
] ; 


const products: Product[] = [
    {
        id: 4,
        title: "컴퓨터",
        price: 10000,
        category: "electronics",
        stock: 40,
        discountRate: 0.2
    },
    {
        id: 2,
        title: "티셔츠",
        price: 20000,
        category: "clothing",
        stock: 100
        // discountRate 없음 → 정가 적용 예정
    },
]

console.log(`회원 ${users.length}명, 상품 ${products.length}개 생성 완료`);
console.table(users);
console.table(products);

회원 3명, 상품 2개 생성 완료
┌─────────┬────┬──────────┬─────────────────────┬────────────┬──────────┐
│ (index) │ id │ name     │ email               │ role       │ isActive │
├─────────┼────┼──────────┼─────────────────────┼────────────┼──────────┤
│ 0       │ 1  │ '관리자' │ 'admin@shop.com'    │ 'admin'    │ true     │
│ 1       │ 2  │ '고객'   │ 'customer@shop.com' │ 'customer' │ true     │
│ 2       │ 3  │ '게스트' │ 'guest@shop.com'    │ 'guest'    │ true     │
└─────────┴────┴──────────┴─────────────────────┴────────────┴──────────┘
┌─────────┬────┬──────────┬───────┬───────────────┬───────┬──────────────┐
│ (index) │ id │ title    │ price │ category      │ stock │ discountRate │
├─────────┼────┼──────────┼───────┼───────────────┼───────┼──────────────┤
│ 0       │ 4  │ '컴퓨터' │ 10000 │ 'electronics' │ 40    │ 0.2          │
│ 1       │ 2  │ '티셔츠' │ 20000 │ 'clothing'    │ 100   │              │
└─────────┴────┴──────────┴───────┴───────────────┴───────┴──────────────┘


In [4]:
// // UserRole에 없는 값이라 거부
// const users: User = {
//     id: 3,
//     name: "게스트",
//     email: "guest@shop.com",
//     role: "manager",  
//     isActive: true
// }

// [타입 안정성 검증] role에 UserRole에 없는 값("manager")을 넣으면 컴파일 단계에서 차단  
// 5:5 - Type '"manager"' is not assignable to type 'UserRole'.

## [심화]
- any를 엄격히 제한한 타입 안전 검색 및 가격 계산 유틸리티 함수 작성 

In [5]:
// 권한 기반 필터 함수 구현

function filterUsersByRole(users: User[], targetRole: UserRole) : User[] {
    return users.filter((user) => user.role === targetRole) ;
}

In [6]:
// 총 결제 금액 계산 함수 구현

function calculateTotalPrice(products: Product[]): number {
    let total = 0 ;

    for (const product of products) {
        const rate = product.discountRate ?? 0 ; // 할인율이 없으면(undefined) 0으로 대체
        total += product.price * (1- rate) ; // 할인 적용 금액을 합계에 더함
    }
    return total ;
}

In [7]:
// 함수 실행 및 결과 확인

// 1) 권한 기반 필터
const customers: User[] = filterUsersByRole(users, "customer");
console.groupCollapsed("customer 역할 회원") ;
console.table(customers) ;
console.groupEnd(); 

customer 역할 회원
  ┌─────────┬────┬────────┬─────────────────────┬────────────┬──────────┐
  │ (index) │ id │ name   │ email               │ role       │ isActive │
  ├─────────┼────┼────────┼─────────────────────┼────────────┼──────────┤
  │ 0       │ 2  │ '고객' │ 'customer@shop.com' │ 'customer' │ true     │
  └─────────┴────┴────────┴─────────────────────┴────────────┴──────────┘


In [8]:
// 2) 총 결제 금액
const totalPrice : number = calculateTotalPrice(products) ;
console.log(`총 결제 금액: ${totalPrice}원`);
console.log(`총 결제 금액: ${totalPrice.toLocaleString()}원`);

총 결제 금액: 28000원
총 결제 금액: 28,000원


## 타입 안정성 정리
- `any`를 사용하지 않고 모든 매개변수·반환값에 타입을 지정함
- `UserRole`, `Category`를 리터럴 유니언으로 제한해 잘못된 값(예: "manager")을 컴파일 단계에서 차단
- `discountRate?`는 `number | undefined`이므로 `?? 0`으로 기본값을 지정해 `NaN` 계산 오류를 방지
  - `||` 대신 `??`를 사용해 null/undefined만 대체